In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_enron_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
enron_dataset = {'ham':[], 'spam':[]}
for i in range(6):
    directory_ham = "../Dataset/ENRON Spam Dataset/enron"+str(i+1)+"/ham"
    directory_spam = "../Dataset/ENRON Spam Dataset/enron"+str(i+1)+"/spam"

    # Loop through all files in the directory_ham
    for filename in os.listdir(directory_ham):
        # Check if it's a file (not a directory_ham)
        if os.path.isfile(os.path.join(directory_ham, filename)):
            # Open the file and do something with it
            with open(os.path.join(directory_ham, filename), 'r', encoding='utf-8', errors='ignore') as file:
                content = file.read()
                # print(content)
                # print(''.join(content.split('\n')[1:]))
                enron_dataset['ham'].append(''.join(content.split('\n')[1:]))
                # Process the content of the file here
                # print(content)
                # break
    
    # Loop through all files in the directory_spam
    for filename in os.listdir(directory_spam):
        # Check if it's a file (not a directory_spam)
        if os.path.isfile(os.path.join(directory_spam, filename)):
            # Open the file and do something with it
            with open(os.path.join(directory_spam, filename), 'r', encoding='utf-8', errors='ignore') as file:
                content = file.read()
                # print(content)
                # print(''.join(content.split('\n')[1:]))
                enron_dataset['spam'].append(''.join(content.split('\n')[1:]))
                # Process the content of the file here
                # print(content)
                # break

print(len(enron_dataset['ham']))
print(len(enron_dataset['spam']))

16545
17170


In [5]:
enron_dataset = transform_enron_dict(enron_dataset)
enron_dataset.head()

,message,label
0,,ham
1,"gary , production from the high island larger ...",ham
2,- calpine daily gas nomination 1 . doc,ham
3,fyi - see note below - already done .stella- -...,ham
4,fyi .- - - - - - - - - - - - - - - - - - - - -...,ham


In [6]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'enron Dataset_'+'.csv')['URL'].to_list())

In [7]:
enron_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'enron Dataset_'+'.csv')['URL']
enron_dataset['Message Len'] = [len(i) for i in enron_dataset['message']]
enron_dataset.head()

,message,label,Extracted URL,Message Len
0,,ham,NaN,0
1,"gary , production from the high island larger ...",ham,NaN,4170
2,- calpine daily gas nomination 1 . doc,ham,NaN,38
3,fyi - see note below - already done .stella- -...,ham,NaN,1144
4,fyi .- - - - - - - - - - - - - - - - - - - - -...,ham,NaN,1101


In [8]:
enron_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'Enron Websites Analysis'+'.csv')
enron_website_analysis_data = enron_website_analysis_data.drop(columns=['ham', 'spam'])
enron_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://wiseschool.com,wiseschool.com,0,0,-1,0
1,http://endosonic.com,endosonic.com,0,0,-1,0
2,http://otq.hellimnone.com,otq.hellimnone.com,0,0,-1,0
3,http://www.netcollage.com,www.netcollage.com,0,0,-1,0
4,http://bizarre.mainoemstore.com,bizarre.mainoemstore.com,0,0,-1,0


In [9]:
len(enron_website_analysis_data)

3456

In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
enron_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_14344\65006589.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  enron_website_analysis_data.iloc[0][0]


'http://wiseschool.com'

In [12]:
# for row in enron_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = enron_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(enron_website_analysis_data['FQDN'])}
website_data = enron_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
enron_dataset['FQDN'] = fqdn
enron_dataset['Website Size in KB'] = website_size
enron_dataset['Website Textual Content Length'] = text_content_len
enron_dataset['Status Code'] = status_code
enron_dataset['Parked'] = parked

In [16]:
enron_dataset = enron_dataset.replace('', np.nan)
enron_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_14344\256516815.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  enron_dataset = enron_dataset.replace('', np.nan)


,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,NaN,ham,NaN,0,NaN,NaN,NaN,NaN,NaN
1,"gary , production from the high island larger ...",ham,NaN,4170,NaN,NaN,NaN,NaN,NaN
2,- calpine daily gas nomination 1 . doc,ham,NaN,38,NaN,NaN,NaN,NaN,NaN
3,fyi - see note below - already done .stella- -...,ham,NaN,1144,NaN,NaN,NaN,NaN,NaN
4,fyi .- - - - - - - - - - - - - - - - - - - - -...,ham,NaN,1101,NaN,NaN,NaN,NaN,NaN


In [17]:
enron_dataset[(enron_dataset['Extracted URL'].notna()) & (enron_dataset['FQDN'].isna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
enron_dataset[(enron_dataset['Extracted URL'].notna()) & (enron_dataset['FQDN'].notna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
107,yo bro . this is my new email address . save i...,ham,http://www.hotmail.com,160,www.hotmail.com,481957.0,19540.0,200.0,1.0
238,hey ! i ' v almost got the curtains finished ....,ham,http://www.hotmail.com,867,www.hotmail.com,481957.0,19540.0,200.0,1.0
239,- - - - - original message - - - - -from : abb...,ham,http://www.hotmail.com-attl.htmmessage-id:from,3536,www.hotmail.com-attl.htmmessage-id.,NaN,NaN,NaN,NaN
250,daren let ' s look at these this weekend ! lov...,ham,http://dl.www.juno.com/get/tagj,593,dl.www.juno.com,0.0,0.0,-1.0,0.0
251,daren i guess i need help with this too ! love...,ham,http://www.juno.com/upgrade.ifyouareunable,1981,www.juno.com,42316.0,2737.0,200.0,0.0
...,...,...,...,...,...,...,...,...,...
33690,innovative website . wide range of remedy sele...,spam,http://s7.cdy.yourneedsdrive.com/qof/ever,979,s7.cdy.yourneedsdrive.com,0.0,0.0,-1.0,0.0
33691,peruse our site and experience the advantages ...,spam,http://4l.ybl.blitzsalesuper.com/qof/and,859,4l.ybl.blitzsalesuper.com,0.0,0.0,-1.0,0.0
33708,"hello ,did you ejaculate before or within a fe...",spam,http://pockmarked.com/et/?medsnothanks:http,874,pockmarked.com,114.0,0.0,200.0,1.0
33711,i got it earlier than expected and it was wrap...,spam,http://t.awechortle.comrestsassure!yourstuffwi...,788,t.awechortle.comrestsassure!yourstuffwillreach.,NaN,NaN,NaN,NaN


In [26]:
#messages with URL
print(len(enron_dataset[(enron_dataset['Extracted URL'].notna())]), len(enron_dataset[(enron_dataset['Extracted URL'].notna())])/len(enron_dataset))

5922 0.1756488209995551


In [27]:
#spam messages with URL
print(len(enron_dataset[(enron_dataset['Extracted URL'].notna()) & (enron_dataset['label']=='spam')]), len(enron_dataset[(enron_dataset['Extracted URL'].notna()) & (enron_dataset['label']=='spam')])/len(enron_dataset[enron_dataset['label']=='spam']))

4716 0.27466511357018053


In [21]:
#unique FQDN
len(set(enron_dataset[(enron_dataset['FQDN'].notna())]['FQDN']))

3107

In [ ]:
only_unique_live_websites_data = enron_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='ham')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='spam')]))

294
97
197


In [24]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='ham')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='spam')]))

114
19
95


In [25]:
enron_dataset.to_csv('../Dataset/Refined_Enron_Email_Spam_Dataset.csv', index=None, quoting=1, escapechar='\\')